# **Video Processing Week 3: Analisis Tangan & Aplikasi Interaktif**

Minggu ini kita akan mempelajari **analisis tangan dan aplikasi interaktif** menggunakan teknologi AI melalui MediaPipe. Fokus utama adalah deteksi dan tracking tangan serta jari untuk membangun aplikasi interaktif yang dapat merespons gestur tangan secara real-time.

**Tujuan Pembelajaran:**

Di akhir sesi ini kita akan mampu:
- Menggunakan MediaPipe Hand untuk mendeteksi dan melacak tangan secara real-time
- Mengimplementasikan sistem pengenalan gestur sederhana berdasarkan posisi landmark
- Membangun aplikasi penghitung jari dan deteksi gestur dasar
- Mengembangkan proyek Virtual Painter menggunakan gestur tangan sebagai kontrol

**Topik Praktik:**
- **Hand Landmark Detection**: Deteksi dan tracking 21 titik landmark pada tangan
- **Hand Gesture Recognition**: Pengenalan gestur sederhana seperti menghitung jari dan membedakan kepalan vs telapak terbuka
- **Virtual Painter Project**: Aplikasi interaktif untuk menggambar menggunakan gestur jari telunjuk dan mengubah warna/menghapus dengan gestur telapak terbuka

> *This module is inspired by the development of last semester’s materials.* 

## **Hand Landmark Detection**

kita akan melakukan deteksi 21 titik landmark pada satu atau dua tangan melalui webcam secara real-time

In [6]:
import cv2
import mediapipe as mp

# ======================
# INIT MODEL
# ======================
hands_model = mp.solutions.hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

drawer = mp.solutions.drawing_utils

# ======================
# TIP FINGER INDEX
# ======================
TIP_IDS = [4, 8, 12, 16, 20]

# ======================
# FUNCTION: COUNT FINGERS
# ======================
def count_fingers(hand_landmarks):
    fingers = []

    # thumb (horizontal check)
    if hand_landmarks.landmark[TIP_IDS[0]].x < hand_landmarks.landmark[TIP_IDS[0] - 1].x:
        fingers.append(1)
    else:
        fingers.append(0)

    # other fingers (vertical check)
    for i in range(1, 5):
        if hand_landmarks.landmark[TIP_IDS[i]].y < hand_landmarks.landmark[TIP_IDS[i] - 2].y:
            fingers.append(1)
        else:
            fingers.append(0)

    return sum(fingers)

# ======================
# CAMERA
# ======================
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Camera error")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)  # mirror view
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    result = hands_model.process(rgb)

    if result.multi_hand_landmarks:

        for hand in result.multi_hand_landmarks:

            drawer.draw_landmarks(frame, hand, mp.solutions.hands.HAND_CONNECTIONS)

            finger_count = count_fingers(hand)

            cv2.putText(
                frame,
                f"FINGERS: {finger_count}",
                (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (0, 255, 255),
                2
            )

            # gesture logic sederhana
            if finger_count == 0:
                gesture = "FIST"
            elif finger_count == 5:
                gesture = "OPEN HAND"
            elif finger_count == 2:
                gesture = "PEACE"
            else:
                gesture = "UNKNOWN"

            cv2.putText(
                frame,
                f"GESTURE: {gesture}",
                (10, 80),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

    cv2.imshow("Smart Hand Gesture System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
hands_model.close()

>>Program ini digunakan untuk mendeteksi tangan secara real-time menggunakan MediaPipe Hands dan OpenCV, kemudian mengubah hasil deteksi menjadi sistem pengenalan gesture sederhana. Webcam digunakan untuk menangkap video yang kemudian diproses frame per frame dalam format RGB. Landmark tangan yang terdeteksi digunakan untuk menghitung jumlah jari yang terbuka berdasarkan posisi titik-titik ujung jari. Berdasarkan jumlah jari tersebut, sistem dapat mengenali gesture seperti fist, open hand, dan peace. Hasil deteksi ditampilkan secara real-time pada layar berupa jumlah jari dan jenis gesture yang sedang dilakukan pengguna hingga program dihentikan dengan tombol “q”.

## **Hand Gesture Recognition Sederhana**

Kita akan mengimplementasikan pengenalan gestur sederhana dengan menerjemahkan data landmark menjadi informasi gestur seperti menghitung jari yang terangkat dan membedakan antara kepalan tangan dengan telapak terbuka.

### Praktik: **Menghitung jumlah jari yang terangkat**

Untuk menghitung jumlah jari yang terangkat, kita bisa melakukan deteksi ujung dan dasar dari masing-masing jari kemudian membandingkan titiknya secara vertikal:

- jika ujung jari lebih tinggi dari dasar/tengah jarinya, maka jari tersebut terangkat. Secara programatik: `finger_tip.y < finger_base.y`

*Catatan: nilai y=0 dimulai dari atas frame. Artinya, nilai y yang lebih kecil menandakan titik ada di bagian atas dan nilai y yang lebih besar ada di bagian bawah*

In [1]:
import cv2
import mediapipe as mp

# ======================
# INIT MODEL
# ======================
hand_model = mp.solutions.hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

drawer = mp.solutions.drawing_utils
style = mp.solutions.drawing_styles

# ======================
# FINGER RULES
# ======================
FINGER_MAP = [
    (4, 3),    # thumb
    (8, 6),    # index
    (12, 10),  # middle
    (16, 14),  # ring
    (20, 18)   # pinky
]

# ======================
# FUNCTION: COUNT FINGERS
# ======================
def get_finger_count(hand):
    count = 0

    for tip, base in FINGER_MAP:
        if hand.landmark[tip].y < hand.landmark[base].y:
            count += 1

    return count

# ======================
# CAMERA
# ======================
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Camera tidak bisa dibuka")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hand_model.process(rgb)

    if result.multi_hand_landmarks:

        for idx, hand in enumerate(result.multi_hand_landmarks):

            drawer.draw_landmarks(
                frame,
                hand,
                mp.solutions.hands.HAND_CONNECTIONS,
                style.get_default_hand_landmarks_style(),
                style.get_default_hand_connections_style()
            )

            fingers = get_finger_count(hand)

            label = "HAND " + str(idx + 1)

            cv2.putText(
                frame,
                f"{label} | FINGERS: {fingers}",
                (10, 60 + idx * 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 255),
                2
            )

    cv2.putText(
        frame,
        "Finger Detection System Active",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.imshow("Custom Finger Counter", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
hand_model.close()

>>Program ini digunakan untuk mendeteksi tangan dan menghitung jumlah jari yang terbuka secara real-time menggunakan MediaPipe Hands dan OpenCV. Webcam digunakan untuk menangkap video yang kemudian diproses frame per frame dengan mengubahnya ke format RGB agar dapat dibaca oleh model. Landmark tangan yang terdeteksi digunakan untuk membandingkan posisi ujung jari dan sendi tengah untuk menentukan apakah jari dalam keadaan terbuka atau tertutup. Jumlah jari yang terbuka kemudian dihitung dan ditampilkan pada layar secara langsung untuk setiap tangan yang terdeteksi. Hasil akhir berupa visualisasi tangan beserta jumlah jari yang terdeteksi secara real-time hingga program dihentikan dengan tombol “q”.

#### 🔎 Eksplorasi

- Menggunakan logika sederhana yang sudah kita implementasikan, meskipun kita telah mengepalkan tangan kita, program masih membaca ada satu jari yang terangkat. Jari apakah yang menyebabkan hal tersebut? dan ide apa yang kamu punya untuk mengatasi masalah ini?

*Hint: Jempol memiliki orientasi berbeda, mungkin perlu logika terpisah*

In [2]:
#### 🔎 Jawaban Eksplorasi

import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1)
mp_draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    if result.multi_hand_landmarks:
        for handLms in result.multi_hand_landmarks:

            lm = handLms.landmark

            # jari (tanpa jempol dulu biar stabil)
            tips = [8, 12, 16, 20]
            dips = [6, 10, 14, 18]

            count = 0

            for tip, dip in zip(tips, dips):
                if lm[tip].y < lm[dip].y:
                    count += 1

            # jempol (versi simpel)
            if lm[4].x < lm[3].x:
                count += 1

            if count == 0:
                text = "FIST"
            elif count == 5:
                text = "OPEN HAND"
            else:
                text = f"{count} FINGERS"

            mp_draw.draw_landmarks(frame, handLms, mp_hands.HAND_CONNECTIONS)

            cv2.putText(frame, text, (50, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 1,
                        (0, 255, 0), 2)

    cv2.imshow("Finger Counter", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

### **Praktik: Membedakan gestur kepalan tangan VS telapak terbuka**

Untuk membedakan gestur kepalan tangan vs telapak terbuka, kita dapat menganalisis posisi landmark jari-jari relatif terhadap telapak tangan. Berikut adalah tahapan implementasinya:

1. Analisis Posisi Jari
Kita membandingkan posisi ujung jari (tip) dengan sendi tengah (PIP) atau pangkal jari:
- **Jari terangkat**: Ujung jari berada di atas sendi tengah (koordinat y lebih kecil)
- **Jari tertutup**: Ujung jari berada di bawah atau sejajar dengan sendi tengah

2. Logika Penentuan Gestur

**Kepalan Tangan (Fist):**
- Semua ujung jari (index 8, 12, 16, 20) berada di bawah sendi tengahnya
- Jempol (index 4) tertutup ke dalam telapak tangan
- Kondisi: `finger_count == 0` atau semua jari dalam posisi tertutup

**Telapak Terbuka (Open Palm):**
- Semua ujung jari berada di atas sendi tengahnya
- Jempol terangkat dan terpisah dari telapak tangan
- Kondisi: `finger_count >= 4` dan jarak antar jari cukup lebar, atau `tip.y < base.y`

In [14]:
import cv2
import mediapipe as mp

# ======================
# INIT
# ======================
hands = mp.solutions.hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

draw = mp.solutions.drawing_utils

# ======================
# STATE FUNCTION
# ======================
def get_state(hand):
    # hitung jari terbuka
    open_fingers = 0
    check = [(8,6), (12,10), (16,14), (20,18)]

    for tip, base in check:
        if hand.landmark[tip].y < hand.landmark[base].y:
            open_fingers += 1

    # state mapping (beda total dari if-else biasa)
    states = {
        0: "FIST",
        1: "ONE",
        2: "TWO",
        3: "THREE",
        4: "FOUR"
    }

    return states.get(open_fingers, "OPEN PALM"), open_fingers

# ======================
# CAMERA
# ======================
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Camera error")

last_state = "NONE"

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    current_state = "NO HAND"

    if result.multi_hand_landmarks:

        for hand in result.multi_hand_landmarks:

            draw.draw_landmarks(
                frame,
                hand,
                mp.solutions.hands.HAND_CONNECTIONS
            )

            current_state, count = get_state(hand)

            # EVENT DETECTION (ini yang bikin beda)
            if current_state != last_state:
                print("Gesture changed:", current_state)
                last_state = current_state

            cv2.putText(
                frame,
                f"State: {current_state}",
                (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 255),
                2
            )

            cv2.putText(
                frame,
                f"Fingers: {count}",
                (10, 80),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2
            )

    cv2.imshow("Gesture State System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
hands.close()

Gesture changed: FOUR
Gesture changed: FIST


>>Program ini digunakan untuk mendeteksi gesture tangan secara real-time menggunakan MediaPipe Hands dan OpenCV dengan pendekatan berbasis state. Webcam digunakan untuk menangkap video yang kemudian diproses frame per frame dalam format RGB. Landmark tangan digunakan untuk menghitung jumlah jari yang terbuka, kemudian hasilnya dipetakan ke dalam state seperti fist, one, two, hingga open palm menggunakan sistem dictionary. Program juga dilengkapi dengan event detection untuk mendeteksi perubahan gesture dari kondisi sebelumnya sehingga sistem dapat mencatat perubahan secara real-time. Hasil deteksi ditampilkan pada layar berupa state dan jumlah jari yang terbuka hingga program dihentikan dengan tombol “q”.

In [3]:
import cv2
import mediapipe as mp
import pyautogui

# ======================
# INIT MODEL
# ======================
hands = mp.solutions.hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

draw = mp.solutions.drawing_utils

# ======================
# DETECT FINGERS
# ======================
def finger_count(hand):
    tips = [(8,6), (12,10), (16,14), (20,18)]

    count = 0
    for tip, base in tips:
        if hand.landmark[tip].y < hand.landmark[base].y:
            count += 1

    return count

# ======================
# ACTION SYSTEM (INI YANG BEDA)
# ======================
def execute_action(fingers):
    if fingers == 0:
        pyautogui.press("space")   # pause/play
        return "PLAY/PAUSE"

    elif fingers == 1:
        pyautogui.press("down")    # scroll down
        return "SCROLL DOWN"

    elif fingers == 2:
        pyautogui.press("up")      # scroll up
        return "SCROLL UP"

    elif fingers == 5:
        pyautogui.hotkey("alt", "tab")  # switch app
        return "SWITCH APP"

    return "NO ACTION"

# ======================
# CAMERA
# ======================
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Camera error")

last_action = ""

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    action_text = "WAITING..."

    if result.multi_hand_landmarks:

        for hand in result.multi_hand_landmarks:

            draw.draw_landmarks(
                frame,
                hand,
                mp.solutions.hands.HAND_CONNECTIONS
            )

            fingers = finger_count(hand)
            action_text = execute_action(fingers)

    cv2.putText(
        frame,
        f"ACTION: {action_text}",
        (10, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0, 255, 255),
        2
    )

    cv2.imshow("Hand Control System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
hands.close()

>>Program ini digunakan untuk mengubah gesture tangan menjadi kontrol langsung pada komputer menggunakan MediaPipe Hands dan OpenCV. Webcam digunakan untuk menangkap video secara real-time, lalu setiap frame diubah ke format RGB agar bisa diproses oleh model MediaPipe. Setelah itu, sistem mendeteksi landmark tangan dan menghitung jumlah jari yang terbuka berdasarkan perbandingan posisi ujung jari dan sendi tengahnya. Jumlah jari tersebut kemudian dipetakan menjadi aksi tertentu seperti play/pause, scroll up/down, atau switch aplikasi menggunakan library PyAutoGUI. Hasil deteksi dan aksi ditampilkan langsung pada layar video secara real-time sampai pengguna menekan tombol “q” untuk keluar dari program.

## **Proyek Mini: Virtual Painter**

Kita akan membuat sebuah aplikasi Virtual Painter yang memungkinkan pengguna menggambar di layar menggunakan gestur tangan:

- Menggabungkan deteksi tangan dan gestur
- Gestur jari telunjuk terangkat -> menggambar
- Gestur telapak tangan terbuka -> mengubah warna
- Gestur tangan ditutup -> menghapus gambar

In [7]:
import cv2
import numpy as np
import mediapipe as mp

def creative_paint():

    cap = cv2.VideoCapture(0)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(
        max_num_hands=1,
        min_detection_confidence=0.6,
        min_tracking_confidence=0.6
    )
    draw = mp.solutions.drawing_utils

    canvas = np.zeros((h, w, 3), dtype=np.uint8)

    prev = None
    mode = "NORMAL"

    colors = [(0,0,255), (0,255,0), (255,0,0), (0,255,255)]
    color_index = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

        if result.multi_hand_landmarks:

            for hand in result.multi_hand_landmarks:
                draw.draw_landmarks(frame, hand, mp.solutions.hands.HAND_CONNECTIONS)

                index = hand.landmark[8]
                middle = hand.landmark[12]
                thumb = hand.landmark[4]

                x = int(index.x * w)
                y = int(index.y * h)

                # =========================
                # GESTURE MODE SWITCH (beda dari dosen)
                # =========================

                # 1. OPEN PALM = change color
                open_palm = (
                    hand.landmark[8].y < hand.landmark[6].y and
                    hand.landmark[12].y < hand.landmark[10].y and
                    hand.landmark[16].y < hand.landmark[14].y and
                    hand.landmark[20].y < hand.landmark[18].y
                )

                # 2. THUMB UP = clear canvas
                thumb_up = thumb.y < hand.landmark[3].y

                if open_palm:
                    color_index = (color_index + 1) % len(colors)
                    mode = "COLOR CHANGE"

                elif thumb_up:
                    canvas[:] = 0
                    mode = "CLEAR"

                else:
                    mode = "DRAW"

                    if prev is not None:
                        cv2.line(canvas, prev, (x, y), colors[color_index], 8)

                    prev = (x, y)

        else:
            prev = None
            mode = "NO HAND"

        # =========================
        # OUTPUT STYLE (INI YANG BEDA DARI DOSEN)
        # =========================

        output = cv2.addWeighted(frame, 0.5, canvas, 0.9, 0)

        cv2.putText(output, f"MODE: {mode}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

        cv2.putText(output, "OPEN PALM = change color | THUMB UP = clear", (10, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 2)

        cv2.imshow("Creative Hand Paint System", output)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    hands.close()


creative_paint()

>>Program ini digunakan untuk membuat aplikasi menggambar berbasis gesture tangan secara real-time menggunakan MediaPipe Hands dan OpenCV. Webcam digunakan untuk menangkap video yang kemudian diproses untuk mendeteksi landmark tangan. Sistem tidak hanya digunakan untuk menggambar garis pada canvas, tetapi juga memiliki beberapa mode interaksi seperti mengganti warna brush menggunakan gesture open palm dan menghapus canvas menggunakan gesture ibu jari terangkat. Pergerakan jari telunjuk digunakan sebagai kontrol utama untuk menggambar pada canvas virtual. Hasil gambar digabungkan dengan tampilan kamera sehingga terlihat seperti sistem augmented reality, dan program berjalan hingga pengguna menekan tombol “q”.